# BÁO CÁO ĐỒ ÁN CAPSTONE (MÔN HỌC CAP2)
## ĐỀ TÀI: AEROFLOW - HỆ THỐNG PIPELINE VÀ KHO DỮ LIỆU PHÂN TÍCH HIỆU SUẤT CHUYẾN BAY TẠI MỸ NĂM 2026
**Hệ sinh thái áp dụng:** Google Cloud Platform (GCP)

---

### 1. GIỚI THIỆU ĐỀ TÀI & MỤC TIÊU
- **Bài toán:** Phân tích nguyên nhân và mô hình hóa các yếu tố tác động đến hiệu suất bay (trễ chuyến - delays, hủy chuyến - cancellations) tại Hoa Kỳ năm 2026.
- **Dữ liệu nguồn:**
  1. **Chuyến bay:** 5 tháng dữ liệu lịch sử BTS (Bureau of Transportation Statistics) tháng 1-5/2026, tổng hợp từ file CSV gốc.
  2. **Thời tiết:** Dữ liệu thời tiết lịch sử tại các sân bay thực tế (lấy từ Open-Meteo Archive API năm 2026).
  3. **Metadata sân bay:** OpenFlights + OurAirports (hãng bay, sân bay, đường băng, tuyến bay).
- **Công cụ sử dụng trong hệ sinh thái Google Cloud:**
  - **Google Colab:** Lập trình và điều phối toàn bộ luồng pipeline.
  - **Google Cloud Storage (GCS):** Lưu trữ Data Lake thô (Bronze Zone).
  - **Google BigQuery:** Kho dữ liệu phân tích Serverless (Data Warehouse - Silver & Gold Zones).
  - **Looker Studio:** Trực quan hóa dữ liệu (Dashboard Báo cáo).

---

### 2. KIẾN TRÚC LUỒNG DỮ LIỆU (ELT PIPELINE)
```
[Nguồn thô: BTS Flights / OpenFlights / OurAirports / Open-Meteo API] 
       │ 
       ▼ (Upload thủ công + Tải về qua Colab)
[Google Cloud Storage (GCS) Buckets]  <-- Data Lake (Bronze Zone)
       │
       ▼ (Load Job)
[BigQuery Staging Tables]             <-- Dữ liệu thô (Staging Dataset)
       │
       ▼ (BigQuery SQL ELT)
[BigQuery Star Schema Tables]         <-- Kho dữ liệu (Warehouse Dataset - Dim & Fact)
       │
       ▼ (BigQuery SQL View/Table)
[Data Mart: mart_delay_analysis]      <-- Bảng tổng hợp (Gold Zone)
       │
       ▼ (Kết nối trực tiếp)
[Looker Studio Dashboard]             <-- Báo cáo trực quan
```

#### Thiết kế Star Schema (Mô hình hình sao):
- **Fact Tables:** `fact_flights` (chứa các số đo trễ chuyến, hủy chuyến), `fact_crew_assignment`, `fact_passenger_manifest`.
- **Dimension Tables:**
  - `dim_airport`: Thông tin chi tiết sân bay (IATA, City, State, Lat, Lon, Elevation, số đường băng,...).
  - `dim_carrier`: Thông tin các hãng hàng không.
  - `dim_aircraft`: Tuổi thọ và loại máy bay.
  - `dim_date`: Lịch ngày trong năm 2026.
  - `dim_weather`: Thông số thời tiết chuẩn hóa theo ngày tại sân bay.
  - `dim_crew`, `dim_passenger`, `dim_ticket`, `dim_cancellation_reason`: Dữ liệu phụ trợ.

---

### CÀI ĐẶT THƯ VIỆN BỔ TRỢ
Thực thi cell dưới đây để cài đặt các thư viện Python cần thiết trên môi trường Colab.

In [ ]:
# Cài đặt các thư viện hỗ trợ kết nối GCP
!pip install -q db-dtypes google-cloud-storage google-cloud-bigquery requests pandas pyarrow

### KẾT NỐI VÀ MOUNT GOOGLE DRIVE
Chúng ta cần kết nối với Google Drive để Python có thể đọc được thư mục chứa code `src/`.

In [ ]:
from google.colab import drive
import os
import sys

# 1. Kết nối Google Drive
drive.mount('/content/drive')

# 2. Tìm và chuyển thư mục làm việc về thư mục chứa Code của bạn
path_code = "/content/drive/MyDrive/CAP2/Code"
path_cap2 = "/content/drive/MyDrive/CAP2"

if os.path.exists(path_code):
    os.chdir(path_code)
    sys.path.append(path_code)
    print(f"Đã kết nối thành công! Thư mục làm việc hiện tại: {os.getcwd()}")
elif os.path.exists(path_cap2):
    os.chdir(path_cap2)
    sys.path.append(path_cap2)
    print(f"Đã kết nối thành công! Thư mục làm việc hiện tại: {os.getcwd()}")
else:
    print("⚠ CẢNH BÁO: Không tìm thấy thư mục CAP2 trên Drive. Vui lòng kiểm tra lại cấu trúc thư mục.")

### BƯỚC 1: XÁC THỰC TÀI KHOẢN GOOGLE CLOUD
Chạy cell dưới đây để xác thực quyền truy cập GCP của bạn trong Colab. Đăng nhập bằng tài khoản Google đã tạo Project GCP.

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Xác thực GCP thành công!")

### CẤU HÌNH DỰ ÁN GCP
Mã Project ID mặc định của bạn đã được cấu hình tự động dưới đây.

In [ ]:
import sys
import os

# Thiết lập project id
PROJECT_ID = "project-8ef27927-934d-40ca-998"
!gcloud config set project {PROJECT_ID}

### BƯỚC 2: EXTRACTION - TRÍCH XUẤT DỮ LIỆU THỜI TIẾT TỪ API
Hàm này sẽ:
1. Tạo bucket GCS nếu chưa có.
2. Quét các file chuyến bay lịch sử BTS trên GCS (`us_flights_2026_01.csv` đến `05.csv`).
3. Chắt lọc danh sách sân bay duy nhất từ cột Origin/Dest.
4. Tra cứu tọa độ sân bay từ OurAirports.
5. Gọi Open-Meteo Archive API cào thời tiết thật năm 2026 cho từng sân bay (phân lô 50 sân bay/lần gọi).
6. Upload file thời tiết lên GCS tại `raw/Weather/weather_raw_2026.csv`.

In [ ]:
from src.config import GCP_PROJECT_ID, GCS_BUCKET_NAME
print(f"Project ID cấu hình: {GCP_PROJECT_ID}")
print(f"GCS Bucket cấu hình: {GCS_BUCKET_NAME}")

import src.extract as extract
# Chạy pipeline trích xuất thời tiết từ Open-Meteo API
extract.run_extraction_pipeline()

### BƯỚC 3: LOADING - NẠP DỮ LIỆU THÔ VÀO STAGING DATASET TRONG BIGQUERY
Hàm này sẽ:
1. Tạo dataset `staging` và `warehouse` trong BigQuery.
2. Tải metadata tĩnh (OurAirports, OpenFlights) từ URL và đẩy lên GCS.
3. Quét động các file chuyến bay `us_flights_2026_*.csv` trên GCS, nạp file đầu bằng TRUNCATE, các file sau bằng APPEND.
4. Nạp file thời tiết `weather_raw_2026.csv` vào BigQuery.
5. Nạp các bảng tĩnh (Airports, Runways, Countries, Regions, Airlines, Routes) lên BigQuery.

In [ ]:
import src.load as load
# Khởi động nạp staging từ GCS về BigQuery
load.run_loading_pipeline()

### BƯỚC 4: TRANSFORMATION - XÂY DỰNG STAR SCHEMA BẰNG BIGQUERY SQL (ELT)
Tại bước này, hệ thống sẽ thực hiện biến đổi dữ liệu trực tiếp trên BigQuery bằng SQL:
1. Tạo các bảng chiều (`dim_airport`, `dim_date`, `dim_weather`, `dim_carrier`, `dim_aircraft`, `dim_crew`, `dim_cancellation_reason`, `dim_passenger`, `dim_ticket`).
2. Tạo bảng sự kiện chính `fact_flights` năm 2026 (sử dụng đúng tên cột BTS gốc).
3. Tạo bảng `fact_crew_assignment` và `fact_passenger_manifest`.
4. Tạo bảng phân tích `mart_delay_analysis` kết nối tất cả thông tin trễ chuyến với thời tiết sân bay.

In [ ]:
import src.transform as transform
# Thực hiện chạy ELT SQL trong BigQuery
transform.run_transformation_pipeline()

### BƯỚC 5: DATA QUALITY AUDIT - KIỂM ĐỊNH CHẤT LƯỢNG DỮ LIỆU
Chạy kiểm tra chất lượng dữ liệu để xác nhận:
1. Khóa chính không trùng lặp và không null (PK Constraints).
2. Khóa ngoại liên kết chính xác 100% giữa các bảng Fact và Dim (FK Constraints).
3. Ràng buộc miền giá trị hợp lệ (Domain Constraints).

In [ ]:
import src.quality as quality
# Chạy kiểm tra chất lượng dữ liệu
quality.run_quality_pipeline()

### BƯỚC 6: CHẠY LIÊN HOÀN TỰ ĐỘNG (ORCHESTRATOR)
Cell dưới đây gọi bộ điều phối tự động chạy liên hoàn toàn bộ 4 bước pipeline (Extract → Load → Transform → Quality) chỉ với 1 lần bấm.

In [ ]:
import src.orchestrator as orchestrator
# Khởi chạy bộ điều phối liên hoàn tự động
orchestrator.run_automated_orchestrator()

### BƯỚC 7: TRUY VẤN MẪU PHÂN TÍCH (ANALYTICAL QUERIES ON BIGQUERY)
Chúng ta thực hiện truy vấn SQL trực tiếp trên bảng `warehouse.mart_delay_analysis` để kết xuất số liệu phân tích độ trễ năm 2026.

In [ ]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project=PROJECT_ID)

# Truy vấn Top 5 hãng hàng không trễ chuyến trung bình cao nhất năm 2026
query = f"""
SELECT 
    carrier_name,
    SUM(total_flights) AS total_flights,
    ROUND(AVG(avg_dep_delay_minutes), 2) AS avg_delay_minutes
FROM `{PROJECT_ID}.warehouse.mart_delay_analysis`
GROUP BY carrier_name
ORDER BY avg_delay_minutes DESC
LIMIT 5
"""

df = client.query(query).to_dataframe()
print("=== TOP 5 HÃNG BAY CÓ ĐỘ TRỄ TRUNG BÌNH CAO NHẤT 2026 ===")
print(df)

In [ ]:
# Truy vấn Tương quan giữa Tuyết rơi tại sân bay nguồn và Tỷ lệ trễ chuyến bay
query_weather = f"""
SELECT 
    CASE 
        WHEN origin_snowfall = 0 THEN 'Không tuyết'
        WHEN origin_snowfall > 0 AND origin_snowfall <= 2 THEN 'Tuyết rơi nhẹ (<=2cm)'
        ELSE 'Tuyết rơi dày (>2cm)'
    END AS weather_condition,
    SUM(total_flights) AS total_flights,
    ROUND(SUM(total_delayed_departures) / SUM(total_flights) * 100, 2) AS delay_rate_pct,
    ROUND(AVG(avg_dep_delay_minutes), 2) AS avg_delay_minutes
FROM `{PROJECT_ID}.warehouse.mart_delay_analysis`
WHERE origin_snowfall IS NOT NULL
GROUP BY weather_condition
ORDER BY delay_rate_pct DESC
"""

df_weather = client.query(query_weather).to_dataframe()
print("=== TÁC ĐỘNG CỦA TUYẾT RƠI ĐẾN TỶ LỆ TRỄ CHUYẾN 2026 ===")
print(df_weather)

### KẾT LUẬN
Hệ thống đã hoạt động hoàn hảo và sẵn sàng kết nối sang **Looker Studio** để vẽ Dashboard.
Nhóm có thể mở Looker Studio, chọn kết nối Google BigQuery, trỏ tới bảng `mart_delay_analysis` để thực hiện kéo thả biểu đồ báo cáo.